# 01 - Explore the project functions

A playground for the functions in `src/caz/`. Each section calls one group of functions,
first on a tiny hand-made example (so you can predict the answer), then on the real data.

- `caz.geo` - distances
- `caz.openair` - download and read the networks' R data files
- `caz.feasibility` - site selection, duplicates, relocations, usable years, verdicts

Notes:
- Run from the project with `uv run jupyter lab` (the kernel then uses the project's `.venv`).
- Real-data cells use the cache in `data/raw/openair/`, filled by `scripts/feasibility.py`.
  If the cache is missing they download the files (a few hundred KB per network).
- Outputs are stripped by the pre-commit hook, so nothing you print here is committed.
- Notebooks are for exploring; package code never imports from them.

In [ ]:
# Reload edited modules automatically, so changes in src/caz/ show up without a kernel restart.
%load_ext autoreload
%autoreload 2

import datetime as dt
from pathlib import Path

import pandas as pd

from caz import feasibility as f
from caz.geo import haversine_km
from caz.openair import (
    NETWORKS,
    annual_summary_url,
    download,
    metadata_url,
    normalise_metadata,
    read_annual_summary,
    read_metadata,
)

# Project root = first parent folder containing pyproject.toml (works wherever Jupyter started).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw" / "openair"
REFERENCE = ROOT / "data" / "reference"
pd.set_option("display.max_columns", 30, "display.width", 200)
ROOT

## 1. `caz.geo.haversine_km`

Great-circle distance in km. Birmingham to Bristol is about 123 km as the crow flies
(the road distance is about 140 km).

In [ ]:
haversine_km(52.4797, -1.8990, 51.4535, -2.5930)

In [ ]:
# Try your own: 0.01 degrees of latitude is ~1.1 km everywhere;
# 0.01 degrees of longitude shrinks with latitude (cos(lat)).
for lat in [50, 55, 60]:
    print(
        lat,
        round(haversine_km(lat, 0, lat + 0.01, 0), 3),
        round(haversine_km(lat, 0, lat, 0.01), 3),
    )

## 2. Reference data: zones and control cities

Hand-curated CSVs in `data/reference/` (checked by `tests/test_reference_data.py`).

In [ ]:
zones = pd.read_csv(
    REFERENCE / "zones.csv", parse_dates=["charging_start", "decision_date", "start_date_announced"]
)
zones[
    [
        "zone_id",
        "class",
        "decision_date",
        "start_date_announced",
        "charging_start",
        "approx_centre_lat",
        "approx_centre_lon",
    ]
]

In [ ]:
controls = pd.read_csv(REFERENCE / "control_cities.csv")
controls[["city_id", "include", "role", "exclusion_reason"]]

In [ ]:
# Distance from every zone to every included control city (feasibility needs >= 15 km for a donor).
included = controls[controls["include"]]
pd.DataFrame(
    {
        c.city_id: [
            haversine_km(
                z.approx_centre_lat, z.approx_centre_lon, c.approx_centre_lat, c.approx_centre_lon
            )
            for z in zones.itertuples()
        ]
        for c in included.itertuples()
    },
    index=zones["zone_id"],
).round(0)

## 3. `caz.openair`: the networks and their files

`NETWORKS` is ordered by priority: when one instrument is listed in two networks, the first wins.

In [ ]:
for net in NETWORKS.values():
    print(f"{net.name:5s} {metadata_url(net)}")
print(annual_summary_url(NETWORKS["AURN"], 2019))

### 3a. `normalise_metadata` on a hand-made table

Upper-cases site codes, parses dates, turns `"ongoing"` into `NaT` (= still running),
and drops rows without coordinates.

In [ ]:
raw = pd.DataFrame(
    {
        "site_id": ["bir1", "bir1", "xyz"],
        "site_name": ["Birmingham A", "Birmingham A", "No coordinates"],
        "location_type": ["Urban Traffic", "Urban Traffic", "Urban Background"],
        "latitude": [52.4, 52.4, None],
        "longitude": [-1.9, -1.9, None],
        "parameter": ["NO2", "PM10", "NO2"],
        "start_date": ["2010-01-01", "2012-05-01", "2015-01-01"],
        "end_date": ["ongoing", "2020-12-31", "ongoing"],
    }
)
normalise_metadata(raw, "LMAM")

### 3b. Real files: `download`, `read_metadata`, `read_annual_summary`

`download` returns the cached file if it exists (pass `refresh=True` to re-download).
It raises `DownloadError` if the server returns anything that is not an R data file -
for example a maintenance page. Change `NET` to `"SAQN"`, `"LMAM"` or `"AQE"` to look at another network.

In [ ]:
NET = "AURN"
network = NETWORKS[NET]
meta_path = download(metadata_url(network), RAW / NET / "metadata.RData")
metadata = read_metadata(meta_path, network)
print(metadata.shape)
metadata.head()

In [ ]:
# What pollutants and site types are there?
display(metadata["parameter"].value_counts().head(10))
metadata.loc[metadata["parameter"] == "NO2", "location_type"].value_counts()

In [ ]:
YEAR = 2023
path = download(annual_summary_url(network, YEAR), RAW / NET / f"annual_{YEAR}.rds")
annual_one_year = read_annual_summary(path, NET)
annual_one_year.sort_values("mean", ascending=False).head(10)

## 4. `caz.feasibility`: from metadata to sites

### 4a. `site_class` and `no2_sites`

`no2_sites` keeps NO2 rows and collapses several instrument records per site into one row:
earliest start, and end = `NaT` if any record is still open.

In [ ]:
[
    (t, f.site_class(t))
    for t in [
        "Urban Traffic",
        "Suburban Background",
        "Urban Industrial",
        "Rural Background",
        "Kerbside",
    ]
]

In [ ]:
sites_net = f.no2_sites(metadata)
print(len(sites_net), "NO2 sites in", NET)
sites_net.head()

### 4b. Load all available networks

Same steps as the script: metadata and annual summaries for every network,
skipping any network that is unavailable.

In [ ]:
from caz.openair import DownloadError

YEARS = range(2010, 2026)
meta_parts, annual_parts = [], []
for name, net in NETWORKS.items():
    try:
        meta_parts.append(
            read_metadata(download(metadata_url(net), RAW / name / "metadata.RData"), net)
        )
    except DownloadError as e:
        print(f"{name}: unavailable ({e})")
        continue
    for year in YEARS:
        try:
            p = download(annual_summary_url(net, year), RAW / name / f"annual_{year}.rds")
        except DownloadError:
            continue
        annual_parts.append(read_annual_summary(p, name))

all_sites = f.no2_sites(pd.concat(meta_parts, ignore_index=True))
annual_raw = pd.concat(annual_parts, ignore_index=True)
all_sites["network"].value_counts()

### 4c. `merge_duplicate_sites`

Sites in *different* networks within 10 m are the same instrument (even with different codes).
The `mapping` table says which kept record each copy was merged into.

In [ ]:
sites, mapping = f.merge_duplicate_sites(all_sites, list(NETWORKS))
print(len(all_sites), "->", len(sites), "sites after merging")
merged = mapping[
    (mapping["network"] != mapping["canon_network"])
    | (mapping["site_id"] != mapping["canon_site_id"])
]
merged.head(20)

In [ ]:
# The Aberdeen example: one instrument, ABD7 in AURN and ABD3 in SAQN.
all_sites[all_sites["site_name"].str.contains("Aberdeen", case=False)][
    ["network", "site_id", "site_name", "latitude", "longitude", "no2_start", "no2_end"]
]

### 4d. `find_relocations` and `chain_relocations`

A closed site with a new site within 300 m opening at most a year later (or up to 30 days before)
is flagged as a possible relocation. `chain_relocations` groups sites into series via union-find.

In [ ]:
relocations = f.find_relocations(sites)
names = sites.set_index(["network", "site_id"])["site_name"]
relocations.assign(
    old_name=[
        names[k] for k in zip(relocations["old_network"], relocations["old_site_id"], strict=True)
    ],
    new_name=[
        names[k] for k in zip(relocations["new_network"], relocations["new_site_id"], strict=True)
    ],
)

In [ ]:
# Toy example: A -> B -> C is one series; D stands alone.
toy = pd.DataFrame(
    {
        "old_network": ["X", "X"],
        "old_site_id": ["A", "B"],
        "new_network": ["X", "X"],
        "new_site_id": ["B", "C"],
    }
)
f.chain_relocations([("X", "A"), ("X", "B"), ("X", "C"), ("X", "D")], toy)

### 4e. `sites_near`

Pairs each site with every area whose centre is within the radius (default 5 km).
Try changing `RADIUS` to see how the counts move.

In [ ]:
areas = pd.concat(
    [
        pd.DataFrame(
            {
                "area_id": zones["zone_id"],
                "area_type": "zone",
                "lat": zones["approx_centre_lat"],
                "lon": zones["approx_centre_lon"],
            }
        ),
        pd.DataFrame(
            {
                "area_id": included["city_id"],
                "area_type": "control",
                "lat": included["approx_centre_lat"],
                "lon": included["approx_centre_lon"],
            }
        ),
    ],
    ignore_index=True,
)
RADIUS = 5.0
near = f.sites_near(sites, areas, radius_km=RADIUS)
near.groupby(["area_type", "area_id"]).size().rename("sites_within_radius")

## 5. Years and verdicts

### 5a. `usable_years`

A year is usable if data capture >= 75%. Annual data from merged copies is pooled
under the kept record first (as in the script).

In [ ]:
annual = annual_raw.merge(mapping, on=["network", "site_id"])
annual = annual.drop(columns=["network", "site_id"]).rename(
    columns={"canon_network": "network", "canon_site_id": "site_id"}
)
usable = f.usable_years(annual)
usable.head()

In [ ]:
# Year-by-year grid for one zone: '#' = usable year, '.' = data but capture < 75%, blank = no data.
ZONE = "bath"
z = near[near["area_id"] == ZONE]
grid = annual.merge(
    z[["network", "site_id", "site_name", "site_class", "distance_km"]], on=["network", "site_id"]
)
grid["mark"] = grid["capture"].map(lambda c: "#" if c >= f.MIN_CAPTURE else ".")
table = grid.pivot_table(
    index=["site_name", "site_class", "distance_km"], columns="year", values="mark", aggfunc="first"
).fillna("")
print(zones.set_index("zone_id").loc[ZONE, ["decision_date", "charging_start"]].to_string())
table

### 5b. `years_before`, `years_after`, `is_usable`, `verdict`

The calendar year containing the date counts as neither before nor after (it is partly treated).
A site is usable with >= 3 years before `pre_end` and >= 1 year after `post_start`.

In [ ]:
years = [2016, 2017, 2018, 2019, 2020, 2022]
charging = dt.date(2021, 3, 15)
decision = dt.date(2019, 3, 5)
print("before charging:", f.years_before(years, charging))
print("before decision:", f.years_before(years, decision))
print("after charging: ", f.years_after(years, charging))
print("usable vs charging:", f.is_usable(years, charging, charging))
print("usable vs decision:", f.is_usable(years, decision, charging))
[f.verdict(n) for n in range(4)]

In [ ]:
# Per-zone usable counts, recomputed here (compare with data/processed/feasibility_summary.csv).
years_by_site = usable.set_index(["network", "site_id"])["usable_years"]
rows = []
for zone in zones.itertuples():
    zs = near[near["area_id"] == zone.zone_id]
    yrs = [years_by_site.get(k, []) for k in zip(zs["network"], zs["site_id"], strict=True)]
    n = sum(f.is_usable(y, zone.charging_start.date(), zone.charging_start.date()) for y in yrs)
    n_dec = sum(f.is_usable(y, zone.decision_date.date(), zone.charging_start.date()) for y in yrs)
    rows.append(
        {
            "zone": zone.zone_id,
            "sites": len(zs),
            "usable": n,
            "verdict": f.verdict(n),
            "usable_dec": n_dec,
        }
    )
pd.DataFrame(rows)

## 6. The script's outputs

`scripts/feasibility.py` writes these two files; reading them here lets you filter and sort freely.

In [ ]:
PROCESSED = ROOT / "data" / "processed"
summary = pd.read_csv(PROCESSED / "feasibility_summary.csv")
site_rows = pd.read_csv(PROCESSED / "feasibility_sites.csv")
summary

In [ ]:
site_rows[site_rows["area_id"] == "birmingham"][
    [
        "network",
        "site_id",
        "site_name",
        "site_class",
        "distance_km",
        "usable_years",
        "usable_vs_charging",
        "replaces",
        "replaced_by",
        "merged_copies",
    ]
]